In [18]:
### Importation

import subprocess
subprocess.run(["pip", "install", "scikit-dimension", "-q"])

import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

###Définition

def grassberger_procaccia(data, n_pairs=50000, n_epsilons=100):
    N = len(data)
    idx1 = np.random.randint(0, N, n_pairs)
    idx2 = np.random.randint(0, N, n_pairs)
    valid = idx1 != idx2
    idx1, idx2 = idx1[valid], idx2[valid]
    diffs     = data[idx1] - data[idx2]
    
    distances = np.sqrt(np.sum(diffs**2, axis=1))  #Computation of all the distances
    distances = distances[distances > 0]  #we exclude the distances equal to zero
    
    eps_min  = np.percentile(distances, 5)  #before that the distances can be instable (Claude)
    eps_max  = np.percentile(distances, 50)  #after that the data will not be representative
    epsilons = np.logspace(np.log10(eps_min), np.log10(eps_max), n_epsilons)
    
    C     = np.array([np.mean(distances < eps) for eps in epsilons])
    valid_C = C > 0
    slope, _ = np.polyfit(np.log(epsilons[valid_C]), np.log(C[valid_C]), 1)  #we estimate the slope
    
    return slope


def run_and_time(method, data, **kwargs):
    start   = time.time()
    result  = method(data, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

N = 2000

proof_cases    = {}
proof_expected = {}

# Case 1: straight line (FD should be 1)
t = np.random.uniform(0, 10, N)
proof_cases['Straight line']      = np.column_stack([t, 2*t + 1])
proof_expected['Straight line']   = 1.0

# Case 2: line with noise (FD should be slightly > 1)
t = np.random.uniform(0, 10, N)
proof_cases['Noisy line']         = np.column_stack([t, 2*t + 1 + 0.5*np.random.normal(0,1,N)])
proof_expected['Noisy line']      = 1.2  # approximate

# Case 3: smooth surface (FD should be 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Smooth surface']     = np.column_stack([x, y, np.sin(x*y)])
proof_expected['Smooth surface']  = 2.0

# Case 4: noisy surface (FD should be slightly > 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Noisy surface']      = np.column_stack([x, y, np.sin(x*y) + 0.3*np.random.normal(0,1,N)])
proof_expected['Noisy surface']   = 2.3  # approximate

# Case 5: completely random (FD should be 3)
proof_cases['Random (x,y,z)']     = np.random.uniform(0, 5, (N, 3))
proof_expected['Random (x,y,z)']  = 3.0

print("Computing FD with GP on all proof cases...\n")
print(f"{'Case':<20} {'Expected':>10} {'Measured':>10} {'Difference':>12} {'Conclusion'}")
print("-" * 72)

###Tests

for name, data in proof_cases.items():
    fd, _  = run_and_time(grassberger_procaccia, data)
    diff   = fd - proof_expected[name]
    print(f"{name:<20} {proof_expected[name]:>10.1f} {fd:>10.3f} {diff:>+12.3f}")


Computing FD with GP on all proof cases...

Case                   Expected   Measured   Difference Conclusion
------------------------------------------------------------------------
Straight line               1.0      0.932       -0.068
Noisy line                  1.2      1.002       -0.198
Smooth surface              2.0      2.181       +0.181
Noisy surface               2.3      2.227       -0.073
Random (x,y,z)              3.0      2.416       -0.584


In [16]:
### Importation

import subprocess
subprocess.run(["pip", "install", "scikit-dimension", "-q"])

import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

###Définition

def grassberger_procaccia(data, n_pairs=50000, n_epsilons=20):
    N = len(data)
    idx1 = np.random.randint(0, N, n_pairs)
    idx2 = np.random.randint(0, N, n_pairs)
    valid = idx1 != idx2
    idx1, idx2 = idx1[valid], idx2[valid]
    diffs     = data[idx1] - data[idx2]
    
    distances = np.sqrt(np.sum(diffs**2, axis=1))  #Computation of all the distances
    distances = distances[distances > 0]  #we exclude the distances equal to zero
    
    eps_min  = np.percentile(distances, 5)  #before that the distances can be instable (Claude)
    eps_max  = np.percentile(distances, 50)  #after that the data will not be representative
    epsilons = np.logspace(np.log10(eps_min), np.log10(eps_max), n_epsilons)
    
    C     = np.array([np.mean(distances < eps) for eps in epsilons])
    valid_C = C > 0
    slope, _ = np.polyfit(np.log(epsilons[valid_C]), np.log(C[valid_C]), 1)  #we estimate the slope
    
    return slope


def run_and_time(method, data, **kwargs):
    start   = time.time()
    result  = method(data, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

N = 2000

proof_cases    = {}
proof_expected = {}

# Case 1: straight line (FD should be 1)
t = np.random.uniform(0, 10, N)
proof_cases['Straight line']      = np.column_stack([t, 2*t + 1])
proof_expected['Straight line']   = 1.0

# Case 2: line with noise (FD should be slightly > 1)
t = np.random.uniform(0, 10, N)
proof_cases['Noisy line']         = np.column_stack([t, 2*t + 1 + 0.5*np.random.normal(0,1,N)])
proof_expected['Noisy line']      = 1.2  # approximate

# Case 3: smooth surface (FD should be 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Smooth surface']     = np.column_stack([x, y, np.sin(x*y)])
proof_expected['Smooth surface']  = 2.0

# Case 4: noisy surface (FD should be slightly > 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Noisy surface']      = np.column_stack([x, y, np.sin(x*y) + 0.3*np.random.normal(0,1,N)])
proof_expected['Noisy surface']   = 2.3  # approximate

# Case 5: completely random (FD should be 3)
proof_cases['Random (x,y,z)']     = np.random.uniform(0, 5, (N, 3))
proof_expected['Random (x,y,z)']  = 3.0

print("Computing FD with GP on all proof cases...\n")
print(f"{'Case':<20} {'Expected':>10} {'Measured':>10} {'Difference':>12} {'Conclusion'}")
print("-" * 72)

###Tests

for name, data in proof_cases.items():
    fd, _  = run_and_time(grassberger_procaccia, data)
    diff   = fd - proof_expected[name]
    print(f"{name:<20} {proof_expected[name]:>10.1f} {fd:>10.3f} {diff:>+12.3f}")


Computing FD with GP on all proof cases...

Case                   Expected   Measured   Difference Conclusion
------------------------------------------------------------------------
Straight line               1.0      0.930       -0.070
Noisy line                  1.2      1.004       -0.196
Smooth surface              2.0      2.174       +0.174
Noisy surface               2.3      2.221       -0.079
Random (x,y,z)              3.0      2.413       -0.587
